In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
import os
import copy
import random
import math
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from torch_geometric.data import HeteroData, Batch
from torch_geometric.nn import HeteroConv, TransformerConv
from data_utils import haversine

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
  torch.cuda.manual_seed(42)
  torch.cuda.manual_seed_all(42)
np.random.seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")



## Configuration


In [ ]:
args = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'hidden_size': 64,
    'epochs': 100,
    'weight_decay': 1e-4,
    'lr': 2e-3,
    'num_heads': 4,
    'num_layers': 2,
    'dropout': 0.1,
    'batch_size': 32
}

## Data Loading and Preprocessing


In [ ]:
class GraphBuilder:
    """Build and cache heterogeneous graphs for each timestamp"""
    def __init__(self, filepath):
        self.filepath = filepath
        self.spatial_scaler = StandardScaler()
        self.time_scaler = StandardScaler()
        self.df = None
        self.spatial_features = None
        self.spatial_nodes = None
        self.radius_to_idx = None
        self.spatial_edges = None
        self.spatial_edge_weights = None
        self.all_months = None
        
    def load_and_preprocess(self):
        """Load data and create static components"""
        print("Loading data...")
        self.df = pd.read_excel(self.filepath)
        self.df['event_month'] = (self.df['event_time'] * 12).astype(int)
        self.df = self.df.sort_values('event_month').reset_index(drop=True)
        
        # Get all unique months
        self.all_months = sorted(self.df['event_month'].unique())
        
        # Create spatial nodes (static)
        self.spatial_nodes = self.df.groupby('fault_radius').agg({
            'magnitude': 'mean',
            'latitude': 'mean',
            'longitude': 'mean',
            'depth': 'mean'
        }).reset_index()
        
        self.radius_to_idx = {radius: idx for idx, radius in enumerate(self.spatial_nodes['fault_radius'])}
        
        features = self.spatial_nodes[['magnitude', 'latitude', 'longitude', 'depth']].values
        self.spatial_features = torch.FloatTensor(self.spatial_scaler.fit_transform(features))
        
        # Create spatial-spatial edges with weights
        self.spatial_edges, self.spatial_edge_weights = self._create_spatial_edges()
        
        print(f"Loaded {len(self.spatial_nodes)} spatial nodes, {self.spatial_edges.shape[1]} spatial edges, {len(self.all_months)} time steps")
        
    def _create_spatial_edges(self):
        """Create edges between all spatial nodes with distance as weights"""
        n = len(self.spatial_nodes)
        edge_index = [[], []]
        edge_weights = []
        
        for i in range(n):
            for j in range(i+1, n):
                lat1, lon1 = self.spatial_nodes.iloc[i]['latitude'], self.spatial_nodes.iloc[i]['longitude']
                lat2, lon2 = self.spatial_nodes.iloc[j]['latitude'], self.spatial_nodes.iloc[j]['longitude']
                
                dist = haversine(lon1, lat1, lon2, lat2)
                
                # Add edges in both directions
                edge_index[0].extend([i, j])
                edge_index[1].extend([j, i])
                # Add weight for both directions
                edge_weights.extend([dist, dist])
        edge_weights = torch.FloatTensor(edge_weights)
        max_dist = torch.max(edge_weights)
        edge_weights = 1 - (edge_weights / max_dist + 1e-8)  # Normalize to [0, 1] and invert (closer -> higher weight)
        
        return torch.LongTensor(edge_index), edge_weights
    
    def create_time_features(self, months):
        time_features = []
        for month in months:
            sin_month = np.sin(2 * np.pi * (month / 12))
            cos_month = np.cos(2 * np.pi * (month / 12))
            time_features.append([sin_month, cos_month, month / 12])
        
        time_features = np.array(time_features)
        return torch.FloatTensor(self.time_scaler.fit_transform(time_features))
    
    def build_graph_up_to_month(self, cutoff_month, target_month):
        """Build heterogeneous graph with all data up to cutoff_month"""
        # Filter data up to cutoff
        past_df = self.df[self.df['event_month'] <= cutoff_month]
        past_months = [i for i in range(target_month + 1)]
        
        # Create time features
        time_features = self.create_time_features(past_months)
        
        # Create spatial-time edges with earthquake magnitude as edge weight
        st_edge_index = [[], []]
        st_edge_weights = []  
        for _, row in past_df.iterrows():
            if row['fault_radius'] in self.radius_to_idx:
                spatial_idx = self.radius_to_idx[row['fault_radius']]
                time_idx = row['event_month']
                st_edge_index[1].append(spatial_idx)
                st_edge_index[0].append(time_idx)
                st_edge_weights.append(row['magnitude'])
        
        # Normalize spatial-time edge weights
        st_edge_weights = torch.FloatTensor(st_edge_weights)
        st_edge_weights = (st_edge_weights - st_edge_weights.min()) / (st_edge_weights.max() - st_edge_weights.min() + 1e-8)
        
        # Create time edges with weight 1
        time_index = [[], []]
        time_weights = []
        for i in range(len(past_months) - 1):
            time_index[0].append(i)
            time_index[1].append(i + 1)
            time_weights.append(1.0)
        
        st_edges = torch.LongTensor(st_edge_index)
        st_weights = st_edge_weights
        time_index = torch.LongTensor(time_index)
        time_weights = torch.FloatTensor(time_weights)

        # Create HeteroData
        data = HeteroData()

        # 1. Assign Node Features
        data['spatial'].x = self.spatial_features
        data['time'].x = time_features

        # 2. Assign Edges and Weights 
        # Spatial-Spatial
        data['spatial', 'nearby', 'spatial'].edge_index = self.spatial_edges
        data['spatial', 'nearby', 'spatial'].edge_attr = self.spatial_edge_weights.view(-1, 1)

        # Time-Spatial
        data['time', 'event', 'spatial'].edge_index = st_edges
        data['time', 'event', 'spatial'].edge_attr = st_weights.view(-1, 1)

        # Time-Time
        data['time', 'past', 'time'].edge_index = time_index
        data['time', 'past', 'time'].edge_attr = time_weights.view(-1, 1)

        return data
    
    def get_target_edges_for_month(self, target_month):
        """Get positive and negative target edges for a specific month"""
        target_df = self.df[self.df['event_month'] == target_month]
        
        
        # Positive edges
        pos_edges = [[], []]
        for _, row in target_df.iterrows():
            if row['fault_radius'] in self.radius_to_idx:
                spatial_idx = self.radius_to_idx[row['fault_radius']]
                time_idx = target_month
                pos_edges[1].append(spatial_idx)
                pos_edges[0].append(time_idx)
        
        pos_edges = torch.LongTensor(pos_edges) if pos_edges[0] else torch.LongTensor([[], []])
        
        # Negative edges
        if pos_edges.shape[1] > 0:
            neg_edges = self._create_negative_samples(
                pos_edges, len(self.spatial_nodes), target_month
            )
        else:
            neg_edges = torch.LongTensor([[], []])
        
        return pos_edges, neg_edges
    
    def _create_negative_samples(self, pos_edges, num_spatial, target_time_idx):
        """Create negative samples"""
        pos_set = set(zip(pos_edges[0].tolist(), pos_edges[1].tolist()))
        neg_edges = [[], []]
        
        num_neg_needed = pos_edges.shape[1]
        attempts = 0
        max_attempts = num_neg_needed * 10
        
        while len(neg_edges[0]) < num_neg_needed and attempts < max_attempts:
            spatial_idx = np.random.randint(0, num_spatial)
            if (target_time_idx, spatial_idx) not in pos_set and spatial_idx not in neg_edges[1]:
                neg_edges[1].append(spatial_idx)
                neg_edges[0].append(target_time_idx)
            attempts += 1
        
        return torch.LongTensor(neg_edges)
    
    def prepare_training_data(self, train_months, index):
        """Pre-build all training, val and test graphs with caching"""
        import pickle
        import hashlib
        
        list_str = str(train_months) + str(index)
        unique_hash = hashlib.md5(list_str.encode()).hexdigest()[:8] # Short hash
        
        os.makedirs("cache", exist_ok=True)
        
        # Filename: cache/graph_data_2_{start_month}_{end_month}_idx{index}_{hash}.pkl
        filename = (f"cache/graph_data_2_"
                    f"{int(train_months[0])}_{int(train_months[-1])}_"
                    f"idx{index}_{unique_hash}.pkl")

        if os.path.exists(filename):
            print(f"Found cached data! Loading from {filename}...")
            with open(filename, 'rb') as f:
                training_samples = pickle.load(f)
            print(f"Successfully loaded {len(training_samples)} samples.")
            return training_samples

        print("\nPre-building graphs (Cache miss)...")
        training_samples = []
        
        for i in range(index, len(train_months)):
            past_cutoff = train_months[i-1]
            target_month = train_months[i]
            
            # Build graph up to past_cutoff
            data = self.build_graph_up_to_month(past_cutoff, target_month)
            
            # Get target edges
            pos_edges, neg_edges = self.get_target_edges_for_month(target_month)
            
            if pos_edges.shape[1] > 0 and neg_edges.shape[1] > 0:
                training_samples.append((data, pos_edges, neg_edges))
        
        print(f"Prepared {len(training_samples)} samples")
        
        print(f"Saving to cache: {filename}")
        with open(filename, 'wb') as f:
            pickle.dump(training_samples, f)
            
        return training_samples
    
    def convert_edges_to_tensor_format(self, training_samples):
        """Make edge indices and attributes contiguous, PyG Native expects [Source, Dest]"""
        updated_data = []
        for i in range(len(training_samples)):
            hetero_data = training_samples[i][0]
            
            for key in hetero_data.edge_types:
                # Just ensure contiguous memory layout
                hetero_data[key].edge_index = hetero_data[key].edge_index.contiguous()
                
                hetero_data[key].edge_attr = hetero_data[key].edge_attr.contiguous()

            updated_data.append((hetero_data, training_samples[i][1], training_samples[i][2]))
        return updated_data
    
    def prepare_batched_data(self, samples_list, batch_size=32):
        """Prepare mini-batches of graphs"""
        batched_samples = []
        
        for i in range(0, len(samples_list), batch_size):
            batch = samples_list[i:i+batch_size]
            batched_samples.append(batch)
        
        return batched_samples


In [ ]:
filepath = "../data/real_data.xlsx"
# filepath = "../data/synthetic_data.xlsx"
builder = GraphBuilder(filepath)
builder.load_and_preprocess()

# Split months
if filepath == "../data/real_data.xlsx":
    train_cutoff_month = int(26 * 12)
    val_cutoff_month = int(29 * 12)
    
if filepath == "../data/synthetic_data.xlsx":
    train_cutoff_month = int(36 * 12)
    val_cutoff_month = int(43.5 * 12)

train_months = [m for m in builder.all_months if m <= train_cutoff_month]
val_months = [m for m in builder.all_months if train_cutoff_month < m <= val_cutoff_month]
test_months = [m for m in builder.all_months if m > val_cutoff_month]

print(f"\nSplit: Train={len(train_months)} months, Val={len(val_months)} months, Test={len(test_months)} months")

# Pre-build all training samples
training_samples = builder.prepare_training_data(train_months, 1)
val_samples = builder.prepare_training_data(train_months + val_months, len(train_months))
test_samples = builder.prepare_training_data(train_months + val_months + test_months, len(train_months) + len(val_months))

# Convert to Native Tensors 
training_samples = builder.convert_edges_to_tensor_format(training_samples)
val_samples = builder.convert_edges_to_tensor_format(val_samples)
test_samples = builder.convert_edges_to_tensor_format(test_samples)

# Create batched samples
batch_size = args['batch_size']
print(f"\nCreating batches of size {batch_size}...")
training_batches = builder.prepare_batched_data(training_samples, batch_size)
val_batches = builder.prepare_batched_data(val_samples, batch_size)
test_batches = builder.prepare_batched_data(test_samples, batch_size)
print(f"Train batches: {len(training_batches)}, Val batches: {len(val_batches)}, Test batches: {len(test_batches)}")



 ## Heterogeneous Transformer Model


In [ ]:
class HeterogeneousTransformer(torch.nn.Module):
    def __init__(self, spatial_feat_dim=4, temporal_feat_dim=3, 
                 hidden_dim=64, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        
        # 1. Encoders
        self.spatial_enc = nn.Linear(spatial_feat_dim, hidden_dim)
        self.temporal_enc = nn.Linear(temporal_feat_dim, hidden_dim)
        
        # 2. Transformer Body
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            conv_dict = {
                # Spatial-Spatial: Dense connection, weighted by Haversine distance
                # edge_dim=1 tells the layer to look for the edge weight
                ('spatial', 'nearby', 'spatial'): TransformerConv(
                    hidden_dim, hidden_dim // num_heads, heads=num_heads, 
                    dropout=dropout, edge_dim=1
                ),
                
                # Time-Spatial: Sparse connection, weighted by Magnitude
                ('time', 'event', 'spatial'): TransformerConv(
                    hidden_dim, hidden_dim // num_heads, heads=num_heads, 
                    dropout=dropout, edge_dim=1
                ),
                
                # Time-Time: Sparse connection, weight is 1.0
                ('time', 'past', 'time'): TransformerConv(
                    hidden_dim, hidden_dim // num_heads, heads=num_heads, 
                    dropout=dropout, edge_dim=1
                )
            }
            # 'sum' aggregation: Sums up the attention results from different edge types
            self.layers.append(HeteroConv(conv_dict, aggr='sum'))

        # 3. Prediction Head
        self.predict_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), 
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, node_feature, edge_index, target_edges, edge_attr_dict=None):
        # 1. Embed Features
        x = {
            'spatial': self.spatial_enc(node_feature['spatial']),
            'time': self.temporal_enc(node_feature['time'])
        }
        
        # 2. Transformer Layers
        for conv in self.layers:
            x = conv(x, edge_index, edge_attr_dict=edge_attr_dict)
            # Non-linearity and Residuals
            x = {key: F.relu(val) for key, val in x.items()}

        # 3. Predict on Target Edges
        time_idx = target_edges[0]
        space_idx = target_edges[1]
        
        feat_t = x['time'][time_idx]
        feat_s = x['spatial'][space_idx]
        
        # Concatenate and pass through prediction head
        combined = torch.cat([feat_t, feat_s], dim=1)
        return self.predict_head(combined)



## Training and Evaluation Functions


In [ ]:
def train_epoch(model, training_samples, optimizer, criterion):
    model.train()
    total_loss = 0
    num_samples = 0
    device = args['device']
    
    for batch_graphs in training_samples:
        graphs = []
        all_pos_edges = []
        all_neg_edges = []
        
        for (data, pos_edges, neg_edges) in batch_graphs:
            graphs.append(data)
            
            # Negative sampling
            neg_edges = builder._create_negative_samples(
                pos_edges, len(builder.spatial_nodes), pos_edges[0, 0].item()
            )
            all_pos_edges.append(pos_edges)
            all_neg_edges.append(neg_edges)
        
        # Batch all graphs together
        batched_data = Batch.from_data_list(graphs).to(device)
        
        # Combine all target edges with proper offsets
        batch_targets = []
        batch_labels = []
        
        cum_spatial_offset = 0
        cum_time_offset = 0
        
        for idx, (data, pos_edges, neg_edges) in enumerate(zip(graphs, all_pos_edges, all_neg_edges)):
            # Offset indices for this graph in the batch
            pos_offset = pos_edges.clone()
            neg_offset = all_neg_edges[idx].clone()
            
            pos_offset[0] += cum_time_offset
            pos_offset[1] += cum_spatial_offset
            neg_offset[0] += cum_time_offset
            neg_offset[1] += cum_spatial_offset
            
            targets = torch.cat([pos_offset, neg_offset], dim=1)
            labels = torch.cat([
                torch.ones(pos_edges.shape[1]), 
                torch.zeros(neg_edges.shape[1])
            ])
            
            batch_targets.append(targets)
            batch_labels.append(labels)
            
            # Update offsets for next graph
            cum_spatial_offset += data['spatial'].x.shape[0]
            cum_time_offset += data['time'].x.shape[0]
        
        x_target = torch.cat(batch_targets, dim=1).to(device)
        y_true = torch.cat(batch_labels).view(-1, 1).to(device)
        
        # Get pre-computed edge attributes
        edge_attr_dict = {}
        for key in batched_data.edge_types:
            edge_attr_dict[key] = batched_data[key].edge_attr
        
        # Forward pass on batched graph
        pred = model(batched_data.x_dict, batched_data.edge_index_dict, x_target, edge_attr_dict)
        
        loss = criterion(pred, y_true)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(batch_graphs)
        num_samples += len(batch_graphs)
    
    return total_loss / num_samples

@torch.no_grad()
def evaluate(model, all_samples, best_model=None, best_val=0):
    """Evaluate model and return metrics and best model with batching"""
    model.eval()
    device = args['device']
    accs = []
    
    for samples in all_samples:
        all_preds, all_labels = [], []
        
        for batch_graphs in samples:
            graphs = []
            all_pos_edges = []
            all_neg_edges = []
            
            for (data, pos_edges, neg_edges) in batch_graphs:
                graphs.append(data)
                all_pos_edges.append(pos_edges)
                all_neg_edges.append(neg_edges)
            
            # Batch all graphs together
            batched_data = Batch.from_data_list(graphs).to(device)
            
            batch_targets = []
            batch_labels = []
            
            cum_spatial_offset = 0
            cum_time_offset = 0
            
            for idx, (data, pos_edges, neg_edges) in enumerate(zip(graphs, all_pos_edges, all_neg_edges)):
                # Offset indices for this graph in the batch
                pos_offset = pos_edges.clone()
                neg_offset = neg_edges.clone()
                
                pos_offset[0] += cum_time_offset
                pos_offset[1] += cum_spatial_offset
                neg_offset[0] += cum_time_offset
                neg_offset[1] += cum_spatial_offset
                
                # Concatenate pos and neg
                targets = torch.cat([pos_offset, neg_offset], dim=1)
                labels = torch.cat([
                    torch.ones(pos_edges.shape[1]),
                    torch.zeros(neg_edges.shape[1])
                ])
                
                batch_targets.append(targets)
                batch_labels.append(labels)
                
                # Update offsets for next graph
                cum_spatial_offset += data['spatial'].x.shape[0]
                cum_time_offset += data['time'].x.shape[0]
            
            x_target = torch.cat(batch_targets, dim=1).to(device)
            labels = torch.cat(batch_labels).view(-1, 1).to(device)
            
            # Get pre-computed edge attributes
            edge_attr_dict = {}
            for key in batched_data.edge_types:
                edge_attr_dict[key] = batched_data[key].edge_attr
            
            # Forward pass on batched graph
            preds = model(batched_data.x_dict, batched_data.edge_index_dict, x_target, edge_attr_dict)
            
            all_preds.append(preds)
            all_labels.append(labels)
            
        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)
        scores = compute_metrics(all_preds, all_labels)
        accs.append(scores)
        
    if accs[1]["f1"] > best_val:
        best_val = accs[1]["f1"]
        best_model = copy.deepcopy(model)
        
    return accs, best_model, best_val

def compute_metrics(all_preds, all_labels, threshold=0.5):
    """Compute different metrics from aggregated predictions and labels"""
    # Convert to numpy for sklearn metrics (Must move to CPU first)
    preds_proba = torch.sigmoid(all_preds).detach().cpu().numpy()
    labels_np = all_labels.detach().cpu().numpy()
    
    # Binary predictions
    pred_binary = (torch.sigmoid(all_preds) > threshold).float()
    
    # Basic accuracy
    accuracy = (pred_binary == all_labels).float().mean()
    
    # Confusion matrix components
    true_pos = ((pred_binary == 1) & (all_labels == 1)).sum().item()
    false_pos = ((pred_binary == 1) & (all_labels == 0)).sum().item()
    false_neg = ((pred_binary == 0) & (all_labels == 1)).sum().item()
    true_neg = ((pred_binary == 0) & (all_labels == 0)).sum().item()
    
    # Threshold-based metrics
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Probability-based metrics (compute only when both classes are present)
    if np.unique(labels_np).size == 2:
        roc_auc = roc_auc_score(labels_np, preds_proba)
    else:
        roc_auc = 0.0
    
    return {
        'accuracy': accuracy.item(),
        'roc_auc': roc_auc,
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'true_pos': true_pos,
        'false_pos': false_pos,
        'true_neg': true_neg,
        'false_neg': false_neg
    }





## Model Initialization


In [ ]:

model = HeterogeneousTransformer(
    spatial_feat_dim=4,
    temporal_feat_dim=3,
    hidden_dim=args['hidden_size'],
    num_heads=args['num_heads'],
    num_layers=args['num_layers'],
    dropout=args['dropout']
).to(args['device'])

optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'], weight_decay=args['weight_decay'])
criterion = nn.BCEWithLogitsLoss()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {model.__class__.__name__}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")





## Training Loop


In [ ]:


best_model = None
best_val = 0

# Shuffle batches
random.shuffle(training_batches)
random.shuffle(val_batches)
random.shuffle(test_batches)

scores_train = []
train_loss = []
scores_val = []
scores_test = []

print("Starting Training with graph batching...")

for epoch in range(args['epochs']):
    loss = train_epoch(model, training_batches, optimizer, criterion)
    train_loss.append(loss)
    accs, best_model, best_val = evaluate(model, [training_batches, val_batches, test_batches], best_model, best_val)
    scores_train.append(accs[0])
    scores_val.append(accs[1])
    scores_test.append(accs[2])
    
    print(
        f"Epoch {epoch + 1}: loss {round(loss, 5)},"
        f"train {round(accs[0]['f1'] * 100, 2)}%,"
        f"valid {round(accs[1]['f1'] * 100, 2)}%,"
        f"test {round(accs[2]['f1'] * 100, 2)}%"
    )

best_accs, best_model, best_val = evaluate(best_model, [training_batches, val_batches, test_batches])
print(
    f"Best model: "
    f"train {round(best_accs[0]['f1'] * 100, 2)}%,"
    f"valid {round(best_accs[1]['f1'] * 100, 2)}%,"
    f"test {round(best_accs[2]['f1']* 100, 2)}%"
)

os.makedirs('model', exist_ok=True)

model_dict = {
    'model_state_dict': best_model.state_dict(),
    'hyperparams': args
}
torch.save(model_dict, 'model/best_transformer.pth')

print("Model saved to 'model/best_transformer.pth'.")




## Training Graphs


In [ ]:


# Set style for good-quality plots  
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

# Create figure with multiple subplots
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.35)

epochs = range(1, len(train_loss) + 1)

# ============= Plot 1: Training Loss =============
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(epochs, train_loss, 'b-', linewidth=2, label='Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Over Epochs', fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# ============= Plot 2: F1 Score Comparison =============
ax2 = fig.add_subplot(gs[1, :])
train_f1 = [s['f1'] * 100 for s in scores_train]
val_f1 = [s['f1'] * 100 for s in scores_val]
test_f1 = [s['f1'] * 100 for s in scores_test]

ax2.plot(epochs, train_f1, 'b-', linewidth=2, label='Train', marker='o', markersize=3, markevery=10)
ax2.plot(epochs, val_f1, 'g-', linewidth=2, label='Validation', marker='s', markersize=3, markevery=10)
ax2.plot(epochs, test_f1, 'r-', linewidth=2, label='Test', marker='^', markersize=3, markevery=10)

# Mark best validation epoch
best_epoch = np.argmax(val_f1)
ax2.axvline(x=best_epoch+1, color='gray', linestyle='--', alpha=0.5, label=f'Best Val (Epoch {best_epoch+1})')
ax2.scatter([best_epoch+1], [val_f1[best_epoch]], color='g', s=100, zorder=5, marker='*')

ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score (%)')
ax2.set_title('F1 Score Across Datasets', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='best')

# ============= Plot 3: Accuracy =============
ax3 = fig.add_subplot(gs[2, 0])
train_acc = [s['accuracy'] * 100 for s in scores_train]
val_acc = [s['accuracy'] * 100 for s in scores_val]
test_acc = [s['accuracy'] * 100 for s in scores_test]

ax3.plot(epochs, train_acc, 'b-', linewidth=1.5, label='Train', alpha=0.7)
ax3.plot(epochs, val_acc, 'g-', linewidth=1.5, label='Validation', alpha=0.7)
ax3.plot(epochs, test_acc, 'r-', linewidth=1.5, label='Test', alpha=0.7)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Accuracy (%)')
ax3.set_title('Accuracy', fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='best')

# ============= Plot 4: Precision =============
ax4 = fig.add_subplot(gs[2, 1])
train_prec = [s['precision'] * 100 for s in scores_train]
val_prec = [s['precision'] * 100 for s in scores_val]
test_prec = [s['precision'] * 100 for s in scores_test]

ax4.plot(epochs, train_prec, 'b-', linewidth=1.5, label='Train', alpha=0.7)
ax4.plot(epochs, val_prec, 'g-', linewidth=1.5, label='Validation', alpha=0.7)
ax4.plot(epochs, test_prec, 'r-', linewidth=1.5, label='Test', alpha=0.7)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Precision (%)')
ax4.set_title('Precision', fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend(loc='best')

# ============= Plot 5: Recall =============
ax5 = fig.add_subplot(gs[2, 2])
train_rec = [s['recall'] * 100 for s in scores_train]
val_rec = [s['recall'] * 100 for s in scores_val]
test_rec = [s['recall'] * 100 for s in scores_test]

ax5.plot(epochs, train_rec, 'b-', linewidth=1.5, label='Train', alpha=0.7)
ax5.plot(epochs, val_rec, 'g-', linewidth=1.5, label='Validation', alpha=0.7)
ax5.plot(epochs, test_rec, 'r-', linewidth=1.5, label='Test', alpha=0.7)
ax5.set_xlabel('Epoch')
ax5.set_ylabel('Recall (%)')
ax5.set_title('Recall', fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.legend(loc='best')

# Save the figure
plt.savefig('training_results_comprehensive_transformer.pdf', bbox_inches='tight', dpi=300)
# plt.show()





## Best Model Evaluation Graphs


In [ ]:


# ============= Create a second figure: Final Performance Comparison =============
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot comparing final metrics
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
train_final = [best_accs[0]['accuracy']*100, best_accs[0]['precision']*100, 
               best_accs[0]['recall']*100, best_accs[0]['f1']*100]
val_final = [best_accs[1]['accuracy']*100, best_accs[1]['precision']*100, 
             best_accs[1]['recall']*100, best_accs[1]['f1']*100]
test_final = [best_accs[2]['accuracy']*100, best_accs[2]['precision']*100, 
              best_accs[2]['recall']*100, best_accs[2]['f1']*100]

x = np.arange(len(metrics_names))
width = 0.25

bars1 = ax1.bar(x - width, train_final, width, label='Train', color='#4472C4', alpha=0.8)
bars2 = ax1.bar(x, val_final, width, label='Validation', color='#70AD47', alpha=0.8)
bars3 = ax1.bar(x + width, test_final, width, label='Test', color='#ED7D31', alpha=0.8)

ax1.set_ylabel('Score (%)')
ax1.set_title('Best Model Performance Comparison', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim([0, 105])

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

# Confusion matrix visualization for test set
confusion_data = [
    [best_accs[2]['true_neg'], best_accs[2]['false_pos']],
    [best_accs[2]['false_neg'], best_accs[2]['true_pos']]
]

im = ax2.imshow(confusion_data, cmap='Blues', aspect='auto')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(['Negative', 'Positive'])
ax2.set_yticklabels(['Negative', 'Positive'])
ax2.set_xlabel('Predicted Label')
ax2.set_ylabel('True Label')
ax2.set_title('Confusion Matrix (Test Set)', fontweight='bold')

# Add text annotations
for i in range(2):
    for j in range(2):
        text = ax2.text(j, i, confusion_data[i][j],
                       ha="center", va="center", color="black", fontsize=12, fontweight='bold')

plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig('final_performance_comparison_transformer.pdf', bbox_inches='tight', dpi=300)
print("Saved: final_performance_comparison_transformer.pdf")
# plt.show()





## ROC-AUC Graph


In [ ]:

# ============= Create a third figure: ROC AUC comparison =============
fig3, ax = plt.subplots(figsize=(8, 6))

train_auc = [s['roc_auc'] for s in scores_train]
val_auc = [s['roc_auc'] for s in scores_val]
test_auc = [s['roc_auc'] for s in scores_test]

ax.plot(epochs, train_auc, 'b-', linewidth=2, label=f"Train (Best: {max(train_auc):.4f})", 
        marker='o', markersize=3, markevery=10)
ax.plot(epochs, val_auc, 'g-', linewidth=2, label=f"Validation (Best: {max(val_auc):.4f})", 
        marker='s', markersize=3, markevery=10)
ax.plot(epochs, test_auc, 'r-', linewidth=2, label=f"Test (Best: {max(test_auc):.4f})", 
        marker='^', markersize=3, markevery=10)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('Epoch')
ax.set_ylabel('ROC AUC Score')
ax.set_title('ROC AUC Score Across Datasets', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('roc_auc_comparison_transformer.pdf', bbox_inches='tight', dpi=300)
print("Saved: roc_auc_comparison_transformer.pdf")
# plt.show()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Best F1 Scores:")
print(f"  Train: {best_accs[0]['f1']*100:.2f}%")
print(f"  Validation: {best_accs[1]['f1']*100:.2f}%")
print(f"  Test: {best_accs[2]['f1']*100:.2f}%")
print(f"\nBest ROC-AUC: {best_accs[2]['roc_auc']:.4f}")



